In [1]:
import pandas as pd
import glob
import os

# 1. CARGA DE ARCHIVOS
ruta = r'C:\Users\Daniela\Documents\RESULTADOS ENSAYO 2025\*.xlsx'
archivos = glob.glob(ruta)
lista_dfs = []
for f in archivos:
    nombre_archivo = os.path.basename(f)
    info = nombre_archivo.split('_')
    temp_df = pd.read_excel(f)
    temp_df['Ensayo'] = info[0]
    temp_df['Curso'] = info[1].replace('.xlsx', '')
    lista_dfs.append(temp_df)
df_maestro = pd.concat(lista_dfs, ignore_index=True)


# 2. ANONIMIZACIÓN Y LIMPIEZA
def anonimizar_texto(texto):
    if pd.isna(texto) or texto == '':
        return texto
    return str(texto)[0] + '*' * 5
def anonimizar_rut(rut):
    if pd.isna(rut) or rut == '':
        return rut
    rut_str = str(rut)
    return '*' * (len(rut_str) - 3) + rut_str[-3:]
df_maestro['Colegio']   = 'Colegio Protegido'
df_maestro['Nombre']    = df_maestro['Nombre'].apply(anonimizar_texto)
df_maestro['Apellido']  = df_maestro['Apellido'].apply(anonimizar_texto)
df_maestro['Rut']       = df_maestro['Rut'].apply(anonimizar_rut)

if 'Email' in df_maestro.columns:
    df_maestro = df_maestro.drop(columns=['Email'])
df_maestro.columns = df_maestro.columns.str.strip()
columna_objetivo = 'PAES M1'
df_maestro_limpio = df_maestro[df_maestro[columna_objetivo] > 0].copy()


# 3. REPORTE ESTADÍSTICO
ensayos = sorted(df_maestro_limpio['Ensayo'].unique())
tablas_por_ensayo = {}
for ensayo in ensayos:
    df_ensayo = df_maestro_limpio[df_maestro_limpio['Ensayo'] == ensayo]
    tabla = df_ensayo.groupby('Curso')[columna_objetivo].agg(
        Promedio='mean',
        Minimo='min',
        Maximo='max'
    ).round(1).reset_index()
    tablas_por_ensayo[ensayo] = tabla


# 4. SEGUIMIENTO INDIVIDUAL
ensayos_interes = ['E1', 'E2', 'E3']
df_filtrado = df_maestro_limpio[df_maestro_limpio['Ensayo'].isin(ensayos_interes)]

pivot_alumnos = df_filtrado.pivot_table(
    index=['Nombre', 'Apellido', 'Curso'],
    columns='Ensayo',
    values=columna_objetivo,
    aggfunc='mean'
)
if 'E1' in pivot_alumnos.columns and 'E2' in pivot_alumnos.columns:
    pivot_alumnos['Dif_E2_E1'] = (pivot_alumnos['E2'] - pivot_alumnos['E1']).round(0)
if 'E2' in pivot_alumnos.columns and 'E3' in pivot_alumnos.columns:
    pivot_alumnos['Dif_E3_E2'] = (pivot_alumnos['E3'] - pivot_alumnos['E2']).round(0)
pivot_alumnos = pivot_alumnos.reset_index()


# 5. ALERTAS CRÍTICAS: POR CURSO
UMBRAL = 50
alertas = []
comparaciones = [('E1', 'E2', 'Dif_E2_E1'), ('E2', 'E3', 'Dif_E3_E2')]
for _, row in pivot_alumnos.iterrows():
    estudiante = f"{row['Nombre']} {row['Apellido']}"
    curso = row['Curso']
    for e_ant, e_sig, col_dif in comparaciones:
        if col_dif in pivot_alumnos.columns and not pd.isna(row.get(col_dif)):
            dif = int(row[col_dif])
            if abs(dif) >= UMBRAL:
                alertas.append({
                    'Curso': curso,
                    'Estudiante': estudiante,
                    'Movimiento': 'Subió' if dif > 0 else 'Bajó',
                    'Puntos': abs(dif),
                    'Tramo': f'{e_ant} → {e_sig}'
                })

df_alertas = pd.DataFrame(alertas)
if not df_alertas.empty:
    df_alertas['orden_mov'] = df_alertas['Movimiento'].map({'Subió': 0, 'Bajó': 1})
    df_alertas = df_alertas.sort_values(
        ['Curso', 'orden_mov', 'Puntos'],
        ascending=[True, True, False]
    ).drop(columns='orden_mov').reset_index(drop=True)

# 6. EXPORTAR A EXCEL
with pd.ExcelWriter('Reporte_Edumetrika_Final.xlsx', engine='openpyxl') as writer:

    # Una hoja por ensayo con el resumen de los 10 cursos
    for ensayo, tabla in tablas_por_ensayo.items():
        tabla.to_excel(writer, sheet_name=f'Resumen_{ensayo}', index=False)

    # Seguimiento individual E1-E2-E3
    pivot_alumnos.to_excel(writer, sheet_name='Seguimiento_E1_E2_E3', index=False)

    # Alertas críticas ordenadas por curso
    if not df_alertas.empty:
        for curso in sorted(df_alertas['Curso'].unique()):
            df_curso = df_alertas[df_alertas['Curso'] == curso]
            df_curso.to_excel(writer, sheet_name=f'Alertas_{curso}', index=False)

    # Hoja resumen de todas las alertas
    if not df_alertas.empty:
        df_alertas.to_excel(writer, sheet_name='Alertas_Todas', index=False)

n_alertas = len(df_alertas) if not df_alertas.empty else 0
print(f" ¡Listo! Se generaron {n_alertas} alertas críticas.")
print(" Revisa 'Reporte_Edumetrika_Final.xlsx'")
print(f"   → {len(ensayos)} hojas de resumen por ensayo")
print(f"   → 1 hoja de seguimiento individual")
print(f"   → {df_alertas['Curso'].nunique() if not df_alertas.empty else 0} hojas de alertas por curso")
print("Guardando en:", os.path.abspath('Reporte_Edumetrika_Final.xlsx'))

 ¡Listo! Se generaron 325 alertas críticas.
 Revisa 'Reporte_Edumetrika_Final.xlsx'
   → 6 hojas de resumen por ensayo
   → 1 hoja de seguimiento individual
   → 10 hojas de alertas por curso
Guardando en: C:\Users\Daniela\Reporte_Edumetrika_Final.xlsx
